# DeepSets with latent export

Notebook này dùng lại pipeline DeepSets, nhưng thêm:
1. `model.encode(...)` để lấy latent embedding.
2. Xuất `latent_database_cfg_plus_temperature.csv`.
3. Lưu file model final kèm thông tin scaler và tham số.

In [1]:
# =========================
# Cell 1 - Import
# =========================
import os
import re
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
# =========================
# Cell 2 - Config
# =========================
project_dir = Path(r"C:\Users\Admin\Downloads\cfg data")

metadata_file = project_dir / "mapped_cfg_and_metadata" / "metadata_all_materials.csv"
output_dir = project_dir / "nn_output"
atom_cache_dir = output_dir / "atom_cache"

output_dir.mkdir(parents=True, exist_ok=True)
atom_cache_dir.mkdir(parents=True, exist_ok=True)

MAX_ATOMS = 4000
BATCH_SIZE = 4
N_EPOCHS = 300
PATIENCE = 40

ELEMENTS = ["Zr", "Cu", "Al", "Ni", "Ti"]
ELEMENT_TO_ID = {el: i + 1 for i, el in enumerate(ELEMENTS)}

ATOM_FEATURE_DIM = 8   # element_id, x, y, z, energy, force_mag, csym, temperature(optional later handled separately)
USE_TEMPERATURE = True

In [3]:
# =========================
# Cell 3 - Read AtomEye CFG
# =========================
def read_cfg_atom_eye(cfg_path):
    cfg_path = Path(cfg_path)
    with open(cfg_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f if line.strip()]

    H = np.zeros((3, 3), dtype=np.float32)
    for line in lines:
        m = re.match(r"H0\((\d),(\d)\)\s*=\s*([-\d\.Ee+]+)", line)
        if m:
            H[int(m.group(1)) - 1, int(m.group(2)) - 1] = float(m.group(3))

    start_idx = None
    for i, line in enumerate(lines):
        if line.startswith("auxiliary"):
            start_idx = i + 1

    if start_idx is None:
        raise ValueError(f"Cannot find atom data in {cfg_path}")

    elements, frac_coords, energy, csym, forces = [], [], [], [], []

    i = start_idx
    while i < len(lines) - 2:
        try:
            float(lines[i])              # mass
            element = lines[i + 1]
            vals = list(map(float, lines[i + 2].split()))
            if len(vals) < 8:
                i += 1
                continue

            x, y, z = vals[0], vals[1], vals[2]
            c_csym = vals[3]
            c_peratom = vals[4]
            fx, fy, fz = vals[5], vals[6], vals[7]

            elements.append(element)
            frac_coords.append([x, y, z])
            csym.append(c_csym)
            energy.append(c_peratom)
            forces.append([fx, fy, fz])
            i += 3
        except Exception:
            i += 1

    frac_coords = np.asarray(frac_coords, dtype=np.float32)
    forces = np.asarray(forces, dtype=np.float32)
    force_mag = np.linalg.norm(forces, axis=1).astype(np.float32)

    return {
        "elements": np.asarray(elements),
        "frac_coords": frac_coords,
        "energy": np.asarray(energy, dtype=np.float32),
        "csym": np.asarray(csym, dtype=np.float32),
        "force_mag": force_mag,
    }

In [9]:
# =========================
# Cell 4 - Build / load atom cache
# =========================
metadata_df = pd.read_csv(metadata_file)

required_cols = ["material", "cfg_file_mapped", "cfg_path_mapped", "T_cfg", "logtau"]
missing = [c for c in required_cols if c not in metadata_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

metadata_df = metadata_df.dropna(subset=required_cols).reset_index(drop=True)

cache_paths = []

for idx, row in metadata_df.iterrows():
    cache_path = atom_cache_dir / f"sample_{idx:04d}.npy"
    cache_paths.append(str(cache_path))

    if cache_path.exists():
        continue

    d = read_cfg_atom_eye(row["cfg_path_mapped"])

    elements = d["elements"]
    elem_id = np.array([ELEMENT_TO_ID.get(el, 0) for el in elements], dtype=np.float32)
    coords = d["frac_coords"].astype(np.float32)
    energy = d["energy"].reshape(-1, 1)
    force_mag = d["force_mag"].reshape(-1, 1)
    csym = d["csym"].reshape(-1, 1)

    atom_features = np.column_stack([
        elem_id.reshape(-1, 1),
        coords,
        energy,
        force_mag,
        csym
    ]).astype(np.float32)

    np.save(cache_path, atom_features)

metadata_df["atom_cache_path"] = cache_paths
metadata_df.to_csv(output_dir / "nn_metadata_with_cache.csv", index=False)

print("Metadata:", metadata_df.shape)
metadata_df.head()

Metadata: (126, 13)


,material,cfg_file_original,cfg_file_mapped,cfg_path_original,cfg_path_mapped,T_cfg,Tmin_logtau,Tmax_logtau,logtau,logtau_source,relax_file,mapped_file_exists,atom_cache_path
0,Zr46Cu46Al8,dumpZrCuAl.cooldown_638.cfg,Zr46Cu46Al8_638.0K.cfg,C:\Users\Admin\Downloads\cfg data\cfg_files_Zr...,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,638.0,635.36667,1002.03333,15.722426,interpolated,C:\Users\Admin\Downloads\cfg data\relaxation_t...,True,C:\Users\Admin\Downloads\cfg data\nn_output\at...
1,Zr46Cu46Al8,dumpZrCuAl.cooldown_661.cfg,Zr46Cu46Al8_661.0K.cfg,C:\Users\Admin\Downloads\cfg data\cfg_files_Zr...,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,661.0,635.36667,1002.03333,10.618194,interpolated,C:\Users\Admin\Downloads\cfg data\relaxation_t...,True,C:\Users\Admin\Downloads\cfg data\nn_output\at...
2,Zr46Cu46Al8,dumpZrCuAl.cooldown_673.cfg,Zr46Cu46Al8_673.0K.cfg,C:\Users\Admin\Downloads\cfg data\cfg_files_Zr...,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,673.0,635.36667,1002.03333,8.450109,interpolated,C:\Users\Admin\Downloads\cfg data\relaxation_t...,True,C:\Users\Admin\Downloads\cfg data\nn_output\at...
3,Zr46Cu46Al8,dumpZrCuAl.cooldown_678.cfg,Zr46Cu46Al8_678.0K.cfg,C:\Users\Admin\Downloads\cfg data\cfg_files_Zr...,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,678.0,635.36667,1002.03333,7.564629,interpolated,C:\Users\Admin\Downloads\cfg data\relaxation_t...,True,C:\Users\Admin\Downloads\cfg data\nn_output\at...
4,Zr46Cu46Al8,dumpZrCuAl.cooldown_680.cfg,Zr46Cu46Al8_680.0K.cfg,C:\Users\Admin\Downloads\cfg data\cfg_files_Zr...,C:\Users\Admin\Downloads\cfg data\mapped_cfg_a...,680.0,635.36667,1002.03333,7.218741,interpolated,C:\Users\Admin\Downloads\cfg data\relaxation_t...,True,C:\Users\Admin\Downloads\cfg data\nn_output\at...


In [18]:
class CFGAtomDataset(Dataset):
    def __init__(self, df, use_temperature=True):
        self.df = df.reset_index(drop=True)
        self.use_temperature = use_temperature

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        atoms = np.load(row["atom_cache_path"]).astype(np.float32)

        n = min(len(atoms), MAX_ATOMS)
        padded = np.zeros((MAX_ATOMS, atoms.shape[1]), dtype=np.float32)
        mask = np.zeros(MAX_ATOMS, dtype=np.float32)

        padded[:n] = atoms[:n]
        mask[:n] = 1.0

        return {
            "atoms": torch.tensor(padded, dtype=torch.float32),
            "mask": torch.tensor(mask, dtype=torch.float32),
            "temperature": torch.tensor([row["T_cfg"]], dtype=torch.float32),
            "y": torch.tensor([row["logtau"]], dtype=torch.float32),
            "idx": torch.tensor(idx, dtype=torch.long),
        }

In [19]:
# =========================
# Cell 6 - DeepSets model with encode()
# =========================
class DeepSetsRegressor(nn.Module):
    def __init__(self, atom_dim=7, atom_hidden=64, latent_dim=64, rho_hidden=64, dropout=0.2, use_temperature=True):
        super().__init__()
        self.use_temperature = use_temperature

        self.phi = nn.Sequential(
            nn.Linear(atom_dim, atom_hidden),
            nn.ReLU(),
            nn.Linear(atom_hidden, atom_hidden),
            nn.ReLU(),
            nn.Linear(atom_hidden, latent_dim),
            nn.ReLU(),
        )

        head_in = latent_dim + (1 if use_temperature else 0)

        self.rho = nn.Sequential(
            nn.Linear(head_in, rho_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(rho_hidden, rho_hidden // 2),
            nn.ReLU(),
            nn.Linear(rho_hidden // 2, 1),
        )

    def encode(self, atoms, mask):
        h = self.phi(atoms)
        mask_exp = mask.unsqueeze(-1)
        h = h * mask_exp
        z = h.sum(dim=1) / mask_exp.sum(dim=1).clamp(min=1.0)
        return z

    def forward(self, atoms, mask, temperature=None, return_latent=False):
        z = self.encode(atoms, mask)
        if self.use_temperature:
            x = torch.cat([z, temperature], dim=1)
        else:
            x = z
        y = self.rho(x)
        if return_latent:
            return y, z
        return y

In [22]:
# =========================
# Cell 7 - Train utilities
# =========================
def calc_metrics(y_true, y_pred):
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
    }

def train_one_model(train_ds, val_ds, params, use_temperature=True):
    model = DeepSetsRegressor(
        atom_dim=7,
        atom_hidden=params["atom_hidden"],
        latent_dim=params["latent_dim"],
        rho_hidden=params["rho_hidden"],
        dropout=params["dropout"],
        use_temperature=use_temperature
    ).to(device)

    train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False)

    opt = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    loss_fn = nn.MSELoss()

    best_loss = np.inf
    best_state = None
    wait = 0

    for epoch in range(N_EPOCHS):
        model.train()
        for batch in train_loader:
            atoms = batch["atoms"].to(device)
            mask = batch["mask"].to(device)
            temp = batch["temperature"].to(device)
            y = batch["y"].to(device)

            pred = model(atoms, mask, temp if use_temperature else None)
            loss = loss_fn(pred, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                atoms = batch["atoms"].to(device)
                mask = batch["mask"].to(device)
                temp = batch["temperature"].to(device)
                y = batch["y"].to(device)
                pred = model(atoms, mask, temp if use_temperature else None)
                val_losses.append(loss_fn(pred, y).item())

        val_loss = float(np.mean(val_losses))
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_loss


def predict_dataset(model, dataset, use_temperature=True):
    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False
    )

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in loader:
            atoms = batch["atoms"].to(device)
            mask = batch["mask"].to(device)
            y = batch["y"].to(device)

            temp = batch["temperature"].to(device)

            pred = model(
                atoms,
                mask,
                temp if use_temperature else None
            )

            y_true.append(y.detach().cpu().numpy())
            y_pred.append(pred.detach().cpu().numpy())

    y_true = np.vstack(y_true).ravel()
    y_pred = np.vstack(y_pred).ravel()

    return y_true, y_pred

In [23]:
# =========================
# Cell 8 - Hyperparameter search with 5-fold CV
# =========================
param_grid = [
    {"atom_hidden": 64, "latent_dim": 64, "rho_hidden": 64, "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 4},
    {"atom_hidden": 64, "latent_dim": 128, "rho_hidden": 128, "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 4},
    {"atom_hidden": 128, "latent_dim": 64, "rho_hidden": 64, "dropout": 0.3, "lr": 5e-4, "weight_decay": 1e-4, "batch_size": 4},
]

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_records = []
summary_records = []

for use_temperature in [False, True]:
    set_name = "cfg_plus_temperature" if use_temperature else "cfg_only"
    print("Running:", set_name)

    best_params_per_fold = []
    fold_metrics = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(metadata_df), start=1):
        tr_df = metadata_df.iloc[tr_idx].reset_index(drop=True)
        va_df = metadata_df.iloc[va_idx].reset_index(drop=True)

        best_score = np.inf
        best_model = None
        best_params = None
        best_train_ds = None
        best_val_ds = None

        for params in param_grid:
            tr_ds = CFGAtomDataset(
                    tr_df,
                    use_temperature=use_temperature
            )

            va_ds = CFGAtomDataset(
                    va_df,
                    use_temperature=use_temperature
            )

            model, val_loss = train_one_model(tr_ds, va_ds, params, use_temperature=use_temperature)

            if val_loss < best_score:
                best_score = val_loss
                best_model = model
                best_params = params
                best_train_ds = tr_ds
                best_val_ds = va_ds

        y_tr, p_tr = predict_dataset(best_model, best_train_ds, use_temperature=use_temperature)
        y_va, p_va = predict_dataset(best_model, best_val_ds, use_temperature=use_temperature)

        m_tr = calc_metrics(y_tr, p_tr)
        m_va = calc_metrics(y_va, p_va)

        fold_records.append({
            "feature_set": set_name,
            "fold": fold,
            "best_params": json.dumps(best_params),
            "train_r2": m_tr["r2"],
            "train_rmse": m_tr["rmse"],
            "train_mae": m_tr["mae"],
            "test_r2": m_va["r2"],
            "test_rmse": m_va["rmse"],
            "test_mae": m_va["mae"],
        })

        best_params_per_fold.append({"fold": fold, "best_params": best_params})
        fold_metrics.append(m_va)

    fold_df = pd.DataFrame([r for r in fold_records if r["feature_set"] == set_name])

    summary_records.append({
        "feature_set": set_name,
        "model": "DeepSets",
        "best_parameter_per_fold": json.dumps(best_params_per_fold),
        "train_r2_mean": fold_df["train_r2"].mean(),
        "train_rmse_mean": fold_df["train_rmse"].mean(),
        "train_mae_mean": fold_df["train_mae"].mean(),
        "test_r2_mean": fold_df["test_r2"].mean(),
        "test_rmse_mean": fold_df["test_rmse"].mean(),
        "test_mae_mean": fold_df["test_mae"].mean(),
        "test_r2_std": fold_df["test_r2"].std(),
        "test_rmse_std": fold_df["test_rmse"].std(),
        "test_mae_std": fold_df["test_mae"].std(),
    })

fold_metrics_df = pd.DataFrame(fold_records)
summary_df = pd.DataFrame(summary_records)

fold_metrics_df.to_csv(output_dir / "nn_metrics_each_fold.csv", index=False)
summary_df.to_csv(output_dir / "nn_metrics_summary.csv", index=False)

summary_df

Running: cfg_only
Running: cfg_plus_temperature


,feature_set,model,best_parameter_per_fold,train_r2_mean,train_rmse_mean,train_mae_mean,test_r2_mean,test_rmse_mean,test_mae_mean,test_r2_std,test_rmse_std,test_mae_std
0,cfg_only,DeepSets,"[{""fold"": 1, ""best_params"": {""atom_hidden"": 64...",0.885171,2.590090,1.895353,0.920159,2.061693,1.545383,0.019917,0.374427,0.380323
1,cfg_plus_temperature,DeepSets,"[{""fold"": 1, ""best_params"": {""atom_hidden"": 64...",0.932096,1.994543,1.411804,0.958590,1.435751,1.043175,0.017612,0.217644,0.153710


In [24]:
# =========================
# Cell 9 - Train final best DeepSets cfg_plus_temperature
# No scaling + use best params from previous CV
# =========================

set_name = "cfg_plus_temperature"
use_temperature = True

# Lấy best params từ Cell 8 theo mean test_rmse nhỏ nhất
best_params_row = (
    fold_metrics_df[fold_metrics_df["feature_set"] == set_name]
    .groupby("best_params", as_index=False)
    .agg(mean_test_rmse=("test_rmse", "mean"))
    .sort_values("mean_test_rmse", ascending=True)
    .iloc[0]
)

best_params = json.loads(best_params_row["best_params"])

print("Best params:", best_params)
print("Mean CV test RMSE:", best_params_row["mean_test_rmse"])

# Chia train/test cố định
train_df, test_df = train_test_split(
    metadata_df,
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Không scale
train_ds = CFGAtomDataset(
    train_df,
    use_temperature=use_temperature
)

test_ds = CFGAtomDataset(
    test_df,
    use_temperature=use_temperature
)

# Train final model trên train set, validate bằng test set
final_model, _ = train_one_model(
    train_ds,
    test_ds,
    best_params,
    use_temperature=use_temperature
)

# Predict train/test
y_tr, p_tr = predict_dataset(
    final_model,
    train_ds,
    use_temperature=use_temperature
)

y_te, p_te = predict_dataset(
    final_model,
    test_ds,
    use_temperature=use_temperature
)

train_metrics = calc_metrics(y_tr, p_tr)
test_metrics = calc_metrics(y_te, p_te)

print("Train:", train_metrics)
print("Test :", test_metrics)

# Lưu prediction trên test set
test_perf_df = test_df.copy().reset_index(drop=True)
test_perf_df["y_true"] = y_te
test_perf_df["y_pred"] = p_te
test_perf_df["error"] = test_perf_df["y_true"] - test_perf_df["y_pred"]
test_perf_df["abs_error"] = test_perf_df["error"].abs()

test_perf_df = test_perf_df.sort_values(
    ["material", "T_cfg"],
    ascending=[True, True]
).reset_index(drop=True)

test_perf_path = output_dir / "nn_test_predictions_cfg_plus_temperature.csv"
test_perf_df.to_csv(test_perf_path, index=False)

print("Saved test predictions:", test_perf_path)

# Lưu final model
model_path = output_dir / "final_deepsets_cfg_plus_temperature.pt"

torch.save({
    "model_state_dict": final_model.state_dict(),
    "best_params": best_params,
    "feature_set": set_name,
    "use_temperature": use_temperature,
    "atom_input_dim": ATOM_FEATURE_DIM,
    "train_metrics": train_metrics,
    "test_metrics": test_metrics,
}, model_path)

print("Saved final model:", model_path)

Best params: {'atom_hidden': 64, 'latent_dim': 128, 'rho_hidden': 128, 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 4}
Mean CV test RMSE: 1.4357507215238612
Train: {'r2': 0.9678829312324524, 'rmse': np.float64(1.3305650175594816), 'mae': 1.0643949508666992}
Test : {'r2': 0.968102216720581, 'rmse': np.float64(1.5087486088832456), 'mae': 1.2832497358322144}
Saved test predictions: C:\Users\Admin\Downloads\cfg data\nn_output\nn_test_predictions_cfg_plus_temperature.csv
Saved final model: C:\Users\Admin\Downloads\cfg data\nn_output\final_deepsets_cfg_plus_temperature.pt


In [25]:
# =========================
# Cell 10 - Save final model (No Scaling)
# =========================

save_obj = {
    "model_state_dict": final_model.state_dict(),
    "best_params": best_params,
    "use_temperature": True,
    "elements": ELEMENTS,
    "max_atoms": MAX_ATOMS,
    "atom_feature_dim": 7,
}

torch.save(
    save_obj,
    output_dir / "final_deepsets_cfg_plus_temperature_with_latent.pt"
)

print("Saved:", output_dir / "final_deepsets_cfg_plus_temperature_with_latent.pt")

Saved: C:\Users\Admin\Downloads\cfg data\nn_output\final_deepsets_cfg_plus_temperature_with_latent.pt


In [26]:
# =========================
# Cell - Save final metrics + test performance for both input sets
# =========================

final_summary_records = []

for use_temperature in [False, True]:
    set_name = "cfg_plus_temperature" if use_temperature else "cfg_only"

    # lấy best params từ kết quả CV đã chạy ở Cell 8
    best_params_row = (
        fold_metrics_df[fold_metrics_df["feature_set"] == set_name]
        .groupby("best_params", as_index=False)
        .agg(mean_test_rmse=("test_rmse", "mean"))
        .sort_values("mean_test_rmse", ascending=True)
        .iloc[0]
    )

    best_params = json.loads(best_params_row["best_params"])

    train_df, test_df = train_test_split(
        metadata_df,
        test_size=0.2,
        random_state=RANDOM_STATE
    )

    train_ds = CFGAtomDataset(train_df, use_temperature=use_temperature)
    test_ds = CFGAtomDataset(test_df, use_temperature=use_temperature)

    final_model, _ = train_one_model(
        train_ds,
        test_ds,
        best_params,
        use_temperature=use_temperature
    )

    y_train_true, y_train_pred = predict_dataset(
        final_model,
        train_ds,
        use_temperature=use_temperature
    )

    y_test_true, y_test_pred = predict_dataset(
        final_model,
        test_ds,
        use_temperature=use_temperature
    )

    train_metrics = calc_metrics(y_train_true, y_train_pred)
    test_metrics = calc_metrics(y_test_true, y_test_pred)

    final_summary_records.append({
        "feature_set": set_name,
        "model": "DeepSets",
        "best_params": json.dumps(best_params),
        "train_r2": train_metrics["r2"],
        "train_rmse": train_metrics["rmse"],
        "train_mae": train_metrics["mae"],
        "test_r2": test_metrics["r2"],
        "test_rmse": test_metrics["rmse"],
        "test_mae": test_metrics["mae"],
    })

    test_perf_df = test_df.copy().reset_index(drop=True)

    test_perf_df["feature_set"] = set_name
    test_perf_df["model"] = "DeepSets"
    test_perf_df["temperature"] = test_perf_df["T_cfg"]
    test_perf_df["logtau_true"] = y_test_true
    test_perf_df["logtau_pred"] = y_test_pred
    test_perf_df["error"] = test_perf_df["logtau_true"] - test_perf_df["logtau_pred"]
    test_perf_df["abs_error"] = test_perf_df["error"].abs()

    keep_cols = [
        "feature_set",
        "model",
        "material",
        "cfg_file_mapped",
        "cfg_path_mapped",
        "temperature",
        "logtau_true",
        "logtau_pred",
        "error",
        "abs_error",
    ]

    test_perf_df = test_perf_df[keep_cols].sort_values(
        ["material", "temperature"],
        ascending=[True, True]
    ).reset_index(drop=True)

    test_perf_path = output_dir / f"nn_test_performance_{set_name}.csv"
    test_perf_df.to_csv(test_perf_path, index=False)

    print("Saved:", test_perf_path)

final_summary_df = pd.DataFrame(final_summary_records)

final_summary_df = final_summary_df.sort_values(
    ["test_r2", "test_rmse", "test_mae"],
    ascending=[False, True, True]
).reset_index(drop=True)

summary_path = output_dir / "nn_final_metrics_summary.csv"
final_summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
final_summary_df

Saved: C:\Users\Admin\Downloads\cfg data\nn_output\nn_test_performance_cfg_only.csv
Saved: C:\Users\Admin\Downloads\cfg data\nn_output\nn_test_performance_cfg_plus_temperature.csv
Saved: C:\Users\Admin\Downloads\cfg data\nn_output\nn_final_metrics_summary.csv


,feature_set,model,best_params,train_r2,train_rmse,train_mae,test_r2,test_rmse,test_mae
0,cfg_plus_temperature,DeepSets,"{""atom_hidden"": 64, ""latent_dim"": 128, ""rho_hi...",0.959816,1.488312,1.151219,0.963464,1.614713,1.265429
1,cfg_only,DeepSets,"{""atom_hidden"": 64, ""latent_dim"": 128, ""rho_hi...",0.901643,2.328469,1.660255,0.879004,2.938481,1.848940
